# 車輛、人員、動物辨識：Kaggle GPU 測試與訓練

這個筆記本做兩件事：

1. **測試**：用現成的 YOLO 模型，對你上傳的照片和影片做偵測，比較不同大小模型的速度和偵測結果。
2. **訓練（選用）**：如果你有自己標註好的資料集（含 `dataset.yaml`），就在 GPU 上微調模型，產出 `best.pt` 下載回本機使用。

**開始前，在右側 Settings 面板設定：**

- **Accelerator**：選 `GPU T4 x2` 或 `GPU P100`
- **Internet**：開啟（要下載套件和模型）
- **Input**：加入你上傳的 Dataset（照片、影片，或 YOLO 格式的訓練資料）

沒有加入任何資料的話，會自動下載兩張範例圖片來示範。

In [ ]:
!pip install -q ultralytics

In [ ]:
import torch

print("CUDA:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
DEVICE = 0 if torch.cuda.is_available() else "cpu"

## 設定

需要時修改這一格就好。

In [ ]:
from pathlib import Path

INPUT_DIR = Path("/kaggle/input")      # Kaggle 掛載 Dataset 的位置
WORK_DIR = Path("/kaggle/working")     # 放在這裡的檔案會出現在 Output，可以下載

MODELS = ["yolo26n.pt", "yolo26s.pt", "yolo26m.pt"]  # 由小到大：越大越準，也越慢
CONF = 0.4                             # 信心度門檻，低於這個值的框不算
MAX_FRAMES_PER_VIDEO = 300             # 每支影片最多測幾幀，避免跑太久；None 代表整支

TRAIN = True                           # 找到 dataset.yaml 時是否訓練
BASE_MODEL = "yolo26s.pt"              # 訓練的起點模型
EPOCHS = 50
BATCH = 16                             # GPU 記憶體不夠時調小
TRAIN_DEVICE = DEVICE                  # 選 GPU T4 x2 時可改成 [0, 1] 同時用兩張卡

In [ ]:
# 和 repo 裡 visual_inspection/categories.py 相同的分類規則
VEHICLE, PERSON, ANIMAL = "vehicle", "person", "animal"
CATEGORIES = (VEHICLE, PERSON, ANIMAL)
CLASS_TO_CATEGORY = {
    "person": PERSON,
    **{name: VEHICLE for name in ["bicycle", "car", "motorcycle", "bus", "truck", "train", "boat", "airplane"]},
    **{name: ANIMAL for name in ["bird", "cat", "dog", "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe"]},
    VEHICLE: VEHICLE,
    ANIMAL: ANIMAL,
}


def category_of(class_name):
    return CLASS_TO_CATEGORY.get(class_name.strip().lower())


def class_ids_for(names):
    return sorted(i for i, name in names.items() if category_of(name) is not None)

## 找出要測試的照片、影片和訓練資料集

In [ ]:
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
VIDEO_SUFFIXES = {".mp4", ".avi", ".mov", ".mkv"}

files = sorted(p for p in INPUT_DIR.rglob("*") if p.is_file()) if INPUT_DIR.exists() else []
dataset_yamls = [p for p in files if p.name in ("dataset.yaml", "data.yaml")]
dataset_roots = [p.parent for p in dataset_yamls]


def in_dataset(path):
    # 訓練資料集裡的圖片留給訓練，不拿來做測試
    return any(root in path.parents for root in dataset_roots)


images = [p for p in files if p.suffix.lower() in IMAGE_SUFFIXES and not in_dataset(p)]
videos = [p for p in files if p.suffix.lower() in VIDEO_SUFFIXES and not in_dataset(p)]

if not images and not videos:
    from ultralytics.utils.downloads import safe_download

    sample_dir = WORK_DIR / "samples"
    for url in ["https://ultralytics.com/images/bus.jpg", "https://ultralytics.com/images/zidane.jpg"]:
        safe_download(url, dir=sample_dir)
    images = sorted(sample_dir.glob("*.jpg"))
    print("沒有找到上傳的照片或影片，改用範例圖片。")

print(f"照片 {len(images)} 張、影片 {len(videos)} 支")
print("訓練資料集：", [str(p) for p in dataset_yamls] or "無")

## 1. 測試：比較不同大小的模型

每個模型會把標好框的結果存到 `predictions/<模型名稱>/`，最後整理成比較表。

In [ ]:
import time
from collections import Counter

import cv2
import pandas as pd
from ultralytics import YOLO


def video_frames(path, limit):
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    n = 0
    while limit is None or n < limit:
        ok, frame = cap.read()
        if not ok:
            break
        yield fps, frame
        n += 1
    cap.release()


rows = []
for model_name in MODELS:
    model = YOLO(model_name)
    classes = class_ids_for(model.names)
    out_dir = WORK_DIR / "predictions" / Path(model_name).stem
    out_dir.mkdir(parents=True, exist_ok=True)
    counts, stats = Counter(), {"ms": 0.0, "frames": 0}

    def detect(frame):
        result = model.predict(frame, classes=classes, conf=CONF, device=DEVICE, verbose=False)[0]
        stats["ms"] += sum(result.speed.values())
        stats["frames"] += 1
        counts.update(category_of(result.names[int(c)]) for c in result.boxes.cls.tolist())
        return result.plot()

    for path in images:
        cv2.imwrite(str(out_dir / path.name), detect(cv2.imread(str(path))))

    for path in videos:
        writer = None
        for fps, frame in video_frames(path, MAX_FRAMES_PER_VIDEO):
            annotated = detect(frame)
            if writer is None:
                h, w = annotated.shape[:2]
                writer = cv2.VideoWriter(str(out_dir / f"{path.stem}.mp4"), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
            writer.write(annotated)
        if writer is not None:
            writer.release()

    frames = stats["frames"]
    ms = stats["ms"] / frames if frames else 0
    rows.append({
        "model": model_name,
        "frames": frames,
        "ms_per_frame": round(ms, 1),
        "fps": round(1000 / ms, 1) if ms else 0,
        **{f"{c}_detections": counts[c] for c in CATEGORIES},
    })
    print(f"{model_name}: {frames} frames, {ms:.1f} ms/frame")

comparison = pd.DataFrame(rows)
comparison.to_csv(WORK_DIR / "model_comparison.csv", index=False)
comparison

`ms_per_frame` 和 `fps` 是 **Kaggle GPU** 上的速度，在你自己的電腦上（尤其只有 CPU 時）會慢很多。
要即時處理監視器畫面，通常選 `n` 或 `s` 版本。

## 2. 訓練（選用）

需要先上傳 YOLO 格式的資料集，並在根目錄放 `dataset.yaml`，可以參考 repo 裡的 `data/dataset.yaml` 範本。

In [ ]:
import shutil

if TRAIN and dataset_yamls:
    data_yaml = str(dataset_yamls[0])
    model = YOLO(BASE_MODEL)
    model.train(
        data=data_yaml,
        epochs=EPOCHS,
        imgsz=640,
        batch=BATCH,
        device=TRAIN_DEVICE,
        project=str(WORK_DIR / "runs"),
        name="train",
        exist_ok=True,
    )
    best = Path(model.trainer.best)
    shutil.copy(best, WORK_DIR / "best.pt")
    metrics = YOLO(best).val(data=data_yaml, device=DEVICE, project=str(WORK_DIR / "runs"), name="val", exist_ok=True)
    print(f"mAP50: {metrics.box.map50:.3f}   mAP50-95: {metrics.box.map:.3f}")
    print("訓練好的模型：", WORK_DIR / "best.pt")
else:
    print("沒有找到 dataset.yaml（或 TRAIN = False），略過訓練。")

## 下載結果

執行完後到右側 **Output** 下載：

- `predictions/`：每個模型標好框的照片和影片
- `model_comparison.csv`：模型比較表
- `best.pt`：訓練好的模型（有訓練才會有）

在本機用訓練好的模型：

```bash
python -m visual_inspection.detect --source 0 --model best.pt
```

> 想保留結果的話，請用右上角 **Save Version → Save & Run All (Commit)** 執行，
> 這樣輸出會一直保存在這個筆記本的版本裡。